In [ ]:
import numpy as np
from scipy.io import loadmat, savemat
import sys

sys.path.append("..")
from mienc.support import pair_mutual_information, normalise, surrogate
from mienc.corrector import Corrector
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.special import digamma

In [ ]:
N_BIN = 8
N_BIN2 = N_BIN * N_BIN
mat = loadmat(f"/mnt/DATA/Motion_fMRI/Datasets/eso245/eso245_cra_strin_500.mat")[
    "subj_tcs"
][:, :2, 0]
corrector = Corrector(
    N_BIN, mat.shape[0], 200, 50000, mat.shape[0], cache_dir="../cache/", verbose=True
)
corrector.compute_correction()
corrector300 = Corrector(
    N_BIN, 300, 200, 50000, 300, cache_dir="../cache/", verbose=True
)
corrector300.compute_correction()

In [ ]:
rx = normalise(mat[:, 0])
ry = normalise(mat[:, 1])
mat2 = surrogate(np.array([x, y]).T)
x = mat2[:, 0]
y = mat2[:, 1]

In [ ]:
cc = np.corrcoef(x, y)
cc, corrector.correct(pair_mutual_information(x, y, 8)), -0.5 * np.log(
    1 - cc[0, 1] ** 2
)

In [ ]:
x_binned = pd.cut(np.argsort(np.argsort(x)), N_BIN, labels=False)
y_binned = pd.cut(np.argsort(np.argsort(y)), N_BIN, labels=False)
positions = [(a, b) for a, b in zip(x_binned, y_binned)]
n_sparse = Counter(positions)
n = np.zeros((N_BIN, N_BIN))
for a in n_sparse:
    n[a[0], a[1]] = n_sparse[a]

In [ ]:
factors = np.array([-1, 1, 1, -1])
N = 0


def sub_step(q, n):
    global N
    i_s = np.random.choice(N_BIN, 2, replace=False)
    j_s = np.random.choice(N_BIN, 2, replace=False)
    delta = np.random.random() * min(q[i_s[0], j_s[0]], q[i_s[1], j_s[1]])
    ai = np.repeat(i_s, 2)
    aj = np.tile(j_s, 2)
    DH = np.sum(n[ai, aj] * np.log(1 + factors * delta / q[ai, aj]))  # *delta*N_BIN2
    if DH > 0 or np.random.random() < np.exp(DH):
        q[ai, aj] += factors * delta
        N += 1


def entropy(q):
    return -np.sum(q * np.log(q))


def entropy_no_zero(q):
    return -np.sum(q[q > 0] * np.log(q[q > 0]))


def entropy_unbias(n):
    N = np.sum(n)
    N_i, M_i = np.unique(n, return_counts=True)
    return np.sum(M_i * (N_i + 1) * (digamma(N + 3) - digamma(N_i + 2))) / (N + 2)


G = np.full(51, -np.euler_gamma - np.log(2))
G[1:] += np.cumsum(2 / (1 + 2 * np.arange(50)))


def entropy_grassberger(n):
    N = np.sum(n)
    N_i, M_i = np.unique(n.astype(int), return_counts=True)
    return np.sum(M_i * N_i * (np.log(N) - G[N_i // 2])) / N

In [ ]:
q = np.ones((N_BIN, N_BIN)) / N_BIN2
mcs = 500
ex = entropy(np.sum(q, 0))
ey = entropy(np.sum(q, 1))
N = 0
S = np.zeros(mcs)
AQ = np.zeros_like(q)
for j in range(N_BIN2 * 50):
    sub_step(q, n)
for i in tqdm(range(mcs)):
    for j in range(N_BIN2 * 7):
        sub_step(q, n)
    S[i] = entropy(q)
    AQ += q
print(q)
print(N / ((50 + 7 * mcs) * N_BIN2))

In [ ]:
print(N / ((50 + 7 * mcs) * N_BIN2))

In [ ]:
plt.plot(S[:100])
plt.hlines(np.mean(S), *plt.xlim())
plt.show()
plt.imshow(n / 400)
plt.colorbar()
plt.show()
plt.imshow(AQ / mcs)
plt.colorbar()
plt.show()
plt.imshow((AQ / mcs - n / 400))
plt.colorbar()
plt.show()
plt.imshow((AQ / mcs - n / 400) / (n / 400))
plt.colorbar()
plt.show()
plt.imshow(q)
plt.colorbar()
plt.show()

In [ ]:
good = ex + ey - S[:]  # np.random.random(1000)#
good -= np.mean(good)
ac = np.correlate(good, good, mode="full")
ac = ac[ac.size // 2 :]
plt.plot((ac / (good.size - np.arange(good.size)))[:25] / np.var(good))
plt.xlim(plt.xlim())
plt.hlines(0, *plt.xlim())
# plt.yscale("log")

In [ ]:
np.sum(N)

In [ ]:
mS = ex + ey - np.mean(S[:])
plt.hist(ex + ey - S[:], "auto")
plt.ylim(plt.ylim())
plt.vlines(
    pair_mutual_information(x, y, N_BIN), *plt.ylim(), "r", "dashed", label="C++ raw"
)
plt.vlines(
    -0.5 * np.log(1 - cc[0, 1] ** 2),
    *plt.ylim(),
    "k",
    "dashed",
    label="from Correlation"
)
plt.vlines(
    corrector.correct(pair_mutual_information(x, y, N_BIN)),
    *plt.ylim(),
    "r",
    "dotted",
    label="C++ corrected"
)
plt.vlines(
    2 * np.log(N_BIN)
    - entropy_no_zero(n / np.sum(n))
    - (N_BIN**2 - 2 * N_BIN + 1) / 2 / np.sum(n),
    *plt.ylim(),
    "r",
    "solid",
    label="Miller",
    lw=1
)
plt.vlines(
    -(2 * entropy_unbias(np.full(N_BIN, np.sum(n) // N_BIN)) - entropy_unbias(n)),
    *plt.ylim(),
    "y",
    "dashed",
    label="Balanced"
)
plt.vlines(
    2 * entropy_grassberger(np.full(N_BIN, np.sum(n) // N_BIN))
    - entropy_grassberger(n),
    *plt.ylim(),
    "y",
    "dotted",
    label="Grassberger"
)
plt.vlines(mS, *plt.ylim(), "k", "dotted", label="Average MCMC")
plt.legend()
plt.show()

In [ ]:
entropy_unbias(np.full(N_BIN, 50)) - np.log(N_BIN)

In [ ]:
mS, pair_mutual_information(x, y, 8), np.log(N_BIN), entropy_unbias(n)

In [ ]:
from scipy.special import loggamma

-2 * entropy(np.sum(n, 0)) + np.sum(
    (n + 1) * (loggamma(n + 2) - loggamma(400 + N_BIN2 + 1))
) / (400 + N_BIN2)

##????????????????????????????

In [ ]:
xn = normalise(mat[:, 0])
MIC = np.zeros(100)
for i in range(100):
    MIC[i] = corrector300.correct(pair_mutual_information(xn[:300], xn[i : 300 + i], 8))

plt.plot(np.sqrt(1 - np.exp(-2 * MIC)))
plt.xlim(plt.xlim())
plt.hlines(0, *plt.xlim())

In [ ]:
good = xn  # np.random.random(1000)#
good -= np.mean(good)
ac = np.correlate(good, good, mode="full")
ac = ac[ac.size // 2 :] / (good.size - np.arange(good.size))
AC = -0.5 * np.log(1 - ac**2)
plt.plot(np.abs(AC)[:100], label="MI from autocorrelation")
plt.plot(MIC, label="Auto-MI")
plt.xlim(plt.xlim())
plt.hlines(0, *plt.xlim())
plt.yscale("symlog", linthresh=0.001)
plt.legend(loc=0)
plt.show()

In [ ]:
N_BIN = 15
mat = loadmat(
    f"/home/raffaelli/NonLinearity/NonLinearityData/EEG_el_so_BLP_NEW/trimmed_EEG_fixedTime_band7.mat"
)["EEG"][:, :2, 0]
print(mat.shape)
corrector4080 = Corrector(
    N_BIN, 4080, 200, 50000, 4080, cache_dir="../cache/", verbose=True
)
corrector4080.compute_correction()

In [ ]:
xm = normalise(mat[:, 0])
MIC = np.zeros(375)
for i in range(375):
    # MIC[i] = pair_mutual_information(xm[:4080], xm[i:4080+i],N_BIN)
    MIC[i] = corrector4080.correct(
        pair_mutual_information(xm[:4080], xm[i : 4080 + i], N_BIN)
    )
good = xm - np.mean(xm)
ac = np.correlate(good, good, mode="full")
ac = ac[ac.size // 2 :] / (good.size - np.arange(good.size))
AC = -0.5 * np.log(1 - ac**2)
correlation = False
if correlation:
    plt.plot(np.abs(ac)[:375], label="autocorrelation")
    plt.plot(np.sqrt(1 - np.exp(-2 * MIC)), label="From Auto-MI")
    plt.ylabel("Correlation")
else:
    plt.plot(np.abs(AC)[:375], label="MI from autocorrelation")
    plt.plot(MIC, label="Auto-MI")
    plt.ylabel("MI")
plt.xlim(plt.xlim())
plt.hlines(0, *plt.xlim())
plt.yscale("log")
# plt.yscale("symlog", linthresh=0.001)
plt.xlim(right=100)
plt.legend(loc=0)
plt.show()

In [ ]:
plt.hist2d(np.log(AC[:375]), np.log(MIC + 1e-5), 20)
# plt.hist2d(np.log(AC[:375][MIC<8e-3]), np.log(MIC[MIC<8e-3]+1e-5), 20)
plt.xlabel("Corr")
plt.ylabel("MI")
# plt.xscale("log")
# plt.yscale("log")
plt.show()

In [ ]:
from scipy.stats import pearsonr

pearsonr(np.log(AC[:375][MIC < 8e-3]), np.log(MIC[MIC < 8e-3] + 1e-5))

In [ ]:
np.min(MIC[MIC > 0])

In [ ]:
from glob import glob
import re

x = []
y = []
z = []
xx = -0.5 * np.log(1 - (np.arange(200) / 200) ** 2)
for file in glob("../cache/*"):
    N, B = map(int, re.findall("_([\d]+)", file))
    x.append((B**2 - 2 * B + 1) / (2 * N))
    z.append((B**4 - 5 * B**2 + 6 * B - 2 / B) / (6 * N**2))
    cor = np.load(file)
    y.append(cor[0])
    plt.plot(xx, cor)
plt.xlim(right=0.7)
plt.ylim(top=0.7)
plt.plot([0, 1], [0, 1], ":k")

In [ ]:
plt.scatter(x, y - np.array(z) / 2)
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.plot([np.min(x), np.max(x)], [np.min(x), np.max(x)], "k:")
plt.show()
plt.scatter(
    x, np.array(y) - np.array(x) * (1 - 8.91801097e-03) - 0.814638463 * np.power(x, 2)
)
# plt.scatter(x,np.array(y)-np.array(x))
plt.scatter(x, np.array(y) - np.array(x) - np.array(z) / 1.6)
plt.scatter(x, np.array(y) - np.array(x) - np.array(z) / 1.8)
# plt.scatter(x, np.sqrt(x)/1000)
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.plot(plt.xlim(), [0, 0], "k:")
plt.show()
np.mean(np.array(y) - np.array(x) * (1 - 8.91801097e-03) - 0.814638463 * np.power(x, 2))

In [ ]:
np.polyfit(x, np.array(y) - x, 2)

In [ ]:
mat = loadmat(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/SUS/new_spiking_831882777_REGULAR.mat")[
    "spikes"
]

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
mat[::10, 1, 2].shape

In [ ]:
for units in range(6):
    fig, ax = plt.subplots(4,4, figsize=(15,15))
    for i in range(4):
        for j in range(4):
            b = j if j > i else j + 4
            ax[i,j].scatter(normalise(mat[::, i, units]), normalise(mat[::, b, units]), s=1, c=np.arange(mat.shape[0]))
            ax[i,j].set_title(f"{i}-{b}")
            # ax[i,j].set_aspect("equal")
    plt.show()

In [ ]:
mat = loadmat(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/SUS/new_spiking_831882777_REGULAR.mat")[
    "spikes"
]

In [ ]:
mcor = np.zeros(6)
for session in GOOD_SESSIONS:
    mat = loadmat(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/SUS/new_spiking_{session}_REGULAR.mat")["spikes"]
    for j in tqdm(range(mat.shape[1])):
        for i in range(6):
            acorr=np.correlate(mat[:,j,i], mat[:,j,i], "full")
            mcor[i]+=acorr[10000]/np.max(acorr)
plt.plot(2**np.arange(6),mcor/mat.shape[1])

In [ ]:
import os, json, glob
RESULTS_FOLDER = "/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/"
GOOD_SESSIONS = [
    831882777,
    816200189,
    771160300,
    786091066,
    779839471,
    778998620,
    781842082,
    778240327,
    793224716,
    839068429,
    794812542,
    768515987,
    767871931,
    840012044,
    766640955,
    847657808,
]
groups = {}
statsReg = [{} for __ in GOOD_SESSIONS]
for s, session in enumerate(GOOD_SESSIONS):
    for folder in glob.glob(
        os.path.join(RESULTS_FOLDER, f"new_spiking_{session}_REGULAR_bin*")
    ):
        if os.path.isfile(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ):
            with open(os.path.join(folder, "shape.json")) as fp:
                groups[session] = json.load(fp)[1]
            with open(
                os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
            ) as fp:
                tmp = {k: np.array(v) for k, v in json.load(fp).items()}
            statsReg[s] = {
                "Empirical": 1 - tmp["gaussMI"] / tmp["totalMI"],
                "Shadow": 1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"],
            }
sus_results = {}
for data in ["Empirical", "Shadow"]:
    sus_results[data] = [[] for i in range(6)]
    for s in range(len(GOOD_SESSIONS)):
        for i in range(6):
            sus_results[data][i].append(statsReg[s][data][i])

In [ ]:
plt.plot(2**(np.arange(6)/1),mcor/(9+10+16+12+14+13+11+15+11+13+16+12+11+11+14+15)/2-0.06)
plt.boxplot(sus_results["Shadow"], positions=2**(np.arange(6)/1), showmeans=True)
plt.show()

In [ ]:
plt.hist(np.abs(mat[:,:,5].flatten()), bins="auto")
plt.yscale("log")
plt.show()

In [ ]:
np.where(mat==mat.min()), mat[23267,7,0]

In [ ]:
vec = mat[:,:,0].flatten()
vec.sort()
plt.plot(vec, 1-np.arange(len(vec))/len(vec))
plt.yscale("log")
plt.show()

In [ ]:
np.polyfit(np.log10(vec[54000:-54000]), 1-np.arange(len(vec))[54000:-54000]/len(vec), 1)

In [ ]:
from scipy.stats import linregress, norm
linregress(np.log10(vec[54000:-54000]), 1-np.arange(len(vec))[54000:-54000]/len(vec))

In [ ]:
exps = np.empty(mat.shape[1:])
tail = mat.shape[0]//5
for i in range(7):
    for j in range(6):
        exps[i,j] = -np.polyfit(np.log10(sorted(mat[:,i,j]))[tail:-tail], 1-np.arange(mat.shape[0])[tail:-tail]/mat.shape[0],1)[0]

In [ ]:
import shutil
from glob import glob
for file in glob(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/SUS/new_spiking_831882777_REGULAR.mat"):
    mat = 1/loadmat(file)["spikes"]
    shutil.copy(file, f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/SUS/{file.split('/')[-1][:-4]}_old.mat")
    savemat(file, {"spikes": mat})